In [ ]:
# Necessary import
import pickle
import mlflow
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from mlflow.tracking import MlflowClient

In [ ]:
# Local tracking URI
MLFLOW_TRACKING_URI = 'http://127.0.0.1:5000'
# Set the tracking URI as local server
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
# Set a Machine Learning experiment
mlflow.set_experiment("green-taxi-duration")

2022/06/01 12:27:06 INFO mlflow.tracking.fluent: Experiment with name 'green-taxi-duration' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow-models-alexey/1', experiment_id='1', lifecycle_stage='active', name='green-taxi-duration', tags={}>

In [ ]:
# Function for reading the data
def read_dataframe(filename: str):
    # read the parquet file
    df = pd.read_parquet(filename)

    # Feature engineering to create a duration column
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    # Convert durations to minutes
    df.duration = df.duration.dt.total_seconds() / 60
    # Filter durations
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    # Set of categorical features
    categorical = ['PULocationID', 'DOLocationID']
    # Convert ctergorica; features to string
    df[categorical] = df[categorical].astype(str)
    return df # to return the prepared dataframe


# Function for building data dictionaries
def prepare_dictionaries(df: pd.DataFrame):
    # Create a trajet feature
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    # Set of features
    categorical = ['PU_DO'] # categorical
    numerical = ['trip_distance'] # numerical
    # Build the data dictionaries
    dicts = df[categorical + numerical].to_dict(orient = 'records')
    return dicts # to return dictionaries

In [ ]:
# Read the data
df_train = read_dataframe('data/green_tripdata_2021-01.parquet') # train
df_val = read_dataframe('data/green_tripdata_2021-02.parquet') # validation

# Set the target variable
target = 'duration'
# Train target vector
y_train = df_train[target].values
# Validation target vector
y_val = df_val[target].values

# Prepare dictionaries
dict_train = prepare_dictionaries(df_train) # train
dict_val = prepare_dictionaries(df_val) # validation

In [ ]:
# Start an experiment run
with mlflow.start_run():
    # Set of model parameters
    params = dict(max_depth = 20, n_estimators = 100, min_samples_leaf = 10,
                  random_state = 0)
    # Log the model parameters
    mlflow.log_params(params)

    # Build a model pipeline
    pipeline = make_pipeline(
        DictVectorizer(),
        RandomForestRegressor(**params, n_jobs = -1)
    )

    # Model training
    pipeline.fit(dict_train, y_train)
    # Prediction on validation data
    y_pred = pipeline.predict(dict_val)

    # RMSE score
    rmse = mean_squared_error(y_pred, y_val, squared = False)
    # Print parameters and rmse score
    print(params, rmse)
    # Log the model rmse
    mlflow.log_metric('rmse', rmse)

    # Log the model
    mlflow.sklearn.log_model(pipeline, artifact_path = "model")

{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 0} 15.136777093556063


In [ ]:
# Run ID
RUN_ID = 'b4d3bca8aa8e46a6b8257fe4541b1136'
# Set an MLFlow client
client = MlflowClient(tracking_uri = MLFLOW_TRACKING_URI)

In [ ]:
# Download the artifact containing the vectorizer
path = client.download_artifacts(run_id = RUN_ID, path = 'dict_vectorizer.bin')

In [ ]:
# Load the dictionary vectorizer
with open(path, 'rb') as f_out:
    dv = pickle.load(f_out)

In [ ]:
# Dictionary vectorizer object
dv

DictVectorizer()

---